## Importing Libraries

In [2]:
import glob
import json
from tqdm import tqdm
import random
import os
from groq import Groq

## Setup Files

In [3]:
GROQ_KEY = os.getenv("GROQ_API_KEY")
PATIENT_PROFILES = glob.glob("./patient_profiles/patient_*.json")

PERFECT_GEN_PROMPT = "./prompts/perfect-diary_gen-prompt.txt"
PERFECT_SYS_PROMPT = "./prompts/perfect-diary_sys-prompt.txt"
INCONSISTANCE_GEN_PROMPT = "./prompts/inconsistancies-int_gen-prompt.txt"
INCONSISTANCE_SYS_PROMPT = "./prompts/inconsistancies-int_sys-prompt.txt"

ANALYSIS_PROMPT = "./prompts/analysis-prompt.txt"
ANALYSIS_SYS_PROMPT = "./prompts/analysis_sys-prompt.txt"

DIARY_TEMPLATE = "./diaries_template/diary_template.txt"
DIARY_EXAMPLES = "./diaries_template/diaries_ex.txt"

LAB_EXAMPLE = "./lab_examples/lab_examples.txt"
LAB_SCHEMA = "./lab_examples/lab_schema.txt"

OUTPUT_DIR = "./outputs/"
OUTPUT_EXP_DIR = "./outputs/diary-gen_experiment"
PERFECT_OUTPUT_FILE = "perfect-diary_patient"
ANALYSIS_OUTPUT_FILE = "analysis_patient"
INCONSISTANCE_OUTPUT_FILE = "inconsistancy-diary_patient"
TRACK_FILE = "parameter_patient"

MODEL = "llama-3.3-70b-versatile" # llama-3.3-70b-versatile, openai/gpt-oss-120b, groq/compound

CLIENT = Groq(api_key=GROQ_KEY)
TEMP = 0.7

## Setup Environment

In [5]:
## Setting evironment
os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isdir(os.path.join(OUTPUT_DIR, path)):
        count += 1
        
os.makedirs(f"{OUTPUT_EXP_DIR}_{count}", exist_ok=True)

## Generating Diaries

In [13]:
count = 19

style_modes = [
    "narrative-dominant",
    "telegraphic-hospital-style",
    "exam-and-imaging-focused",
    "toxicity-focused",
    "psychosocial-emphasis"
]

length_modes = [
    "short",
    "medium",
    "long"
]

pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating synthetic clinical diaries")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(PERFECT_GEN_PROMPT, "r", encoding="utf-8") as perfect_gen_prompt_file, \
         open(PERFECT_SYS_PROMPT, "r", encoding="utf-8") as perfect_sys_prompt_file, \
         open(INCONSISTANCE_GEN_PROMPT, "r", encoding="utf-8") as incons_gen_prompt_file, \
         open(INCONSISTANCE_SYS_PROMPT, "r", encoding="utf-8") as incons_sys_prompt_file, \
         open(DIARY_TEMPLATE, "r", encoding="utf-8") as diary_template_file, \
         open(DIARY_EXAMPLES, "r", encoding="utf-8") as diary_ex_file:
             
        patient_id = int(patient.split('_')[2].split('.')[0])
        
        if patient_id == 27:
            
            print(f"Generating diary for patient {patient_id}")
            
            selected_style = random.choice(style_modes)
            selected_length = random.choice(length_modes)
            
            print(f"Selected stylistic mode for patient {patient_id}: {selected_style}")
            print(f"Selected length mode for patient {patient_id}: {selected_length}")
            
            patient_data = json.load(f)
            base_perfect_gen_prompt = perfect_gen_prompt_file.read()
            perfect_sys_prompt = perfect_sys_prompt_file.read()
            diary_template = diary_template_file.read()
            diary_examples = diary_ex_file.read()
            
            perfect_prompt_w_template = base_perfect_gen_prompt.replace("{{TEMPLATE_TEXT}}", diary_template)
            perfect_prompt_w_patient = perfect_prompt_w_template.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
            perfect_prompt_w_style = perfect_prompt_w_patient.replace("{{STYLISTIC_MODE}}", selected_style)
            perfect_prompt_w_length = perfect_prompt_w_style.replace("{{LENGTH_MODE}}", selected_length)
            perfect_prompt_final = perfect_prompt_w_length.replace("{{DIARIES_TEXT}}", diary_examples)
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": perfect_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": perfect_prompt_final
                    }
                ],
                temperature=TEMP
            )
            perfect_result = completion.choices[0].message.content

            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{perfect_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient_id}.txt")
                
            base_inconsistency_gen_prompt = incons_gen_prompt_file.read()
            inconsistency_sys_prompt = incons_sys_prompt_file.read()
            
            inconsistency_prompt_final = base_inconsistency_gen_prompt.replace("{{CLEAN_DIARY}}", perfect_result)
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": inconsistency_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": inconsistency_prompt_final
                    }
                ],
                temperature=TEMP
            )
            inconsistency_result = completion.choices[0].message.content
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{INCONSISTANCE_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{inconsistency_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{INCONSISTANCE_OUTPUT_FILE}_{patient_id}.txt")
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"Model: {MODEL}\n"
                        f"  Stylistic Mode: {selected_style}\n"
                        f"  Length Mode: {selected_length}\n"
                        f"  Temperature: {TEMP}\n"
                        f"\n"
                        f"Perfect diary system prompt:\n{perfect_sys_prompt}\n"
                        f"\n"
                        f"Perfect diary generation prompt:\n{perfect_prompt_final}\n"
                        f"\n"
                        f"Inconsistency diary system prompt:\n{inconsistency_sys_prompt}\n"
                        f"\n"
                        f"Inconsistency diary generation prompt:\n{inconsistency_prompt_final}\n"
                        )
                print(f"Saved parameters to {OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt")
                
            print("\n")
        else:
            print(f"Skipping patient {patient_id} as it has already been processed.")
        
        pbar.update(1)
    
        
pbar.close()

Generating synthetic clinical diaries:   0%|          | 0/30 [00:00<?, ?it/s]

Processing patient: ./patient_profiles\patient_1.json
Skipping patient 1 as it has already been processed.
Processing patient: ./patient_profiles\patient_10.json
Skipping patient 10 as it has already been processed.
Processing patient: ./patient_profiles\patient_11.json
Skipping patient 11 as it has already been processed.
Processing patient: ./patient_profiles\patient_12.json
Skipping patient 12 as it has already been processed.
Processing patient: ./patient_profiles\patient_13.json
Skipping patient 13 as it has already been processed.
Processing patient: ./patient_profiles\patient_14.json
Skipping patient 14 as it has already been processed.
Processing patient: ./patient_profiles\patient_15.json
Skipping patient 15 as it has already been processed.
Processing patient: ./patient_profiles\patient_16.json
Skipping patient 16 as it has already been processed.
Processing patient: ./patient_profiles\patient_17.json
Skipping patient 17 as it has already been processed.
Processing patient: .

Generating synthetic clinical diaries: 100%|██████████| 30/30 [00:26<00:00,  1.13it/s]

Saved LLM output on ./outputs/diary-gen_experiment_19/inconsistancy-diary_patient_27.txt
Saved parameters to ./outputs/diary-gen_experiment_19/parameter_patient_27.txt


Processing patient: ./patient_profiles\patient_28.json
Skipping patient 28 as it has already been processed.
Processing patient: ./patient_profiles\patient_29.json
Skipping patient 29 as it has already been processed.
Processing patient: ./patient_profiles\patient_3.json
Skipping patient 3 as it has already been processed.
Processing patient: ./patient_profiles\patient_30.json
Skipping patient 30 as it has already been processed.
Processing patient: ./patient_profiles\patient_4.json
Skipping patient 4 as it has already been processed.
Processing patient: ./patient_profiles\patient_5.json
Skipping patient 5 as it has already been processed.
Processing patient: ./patient_profiles\patient_6.json
Skipping patient 6 as it has already been processed.
Processing patient: ./patient_profiles\patient_7.json
Skipping patient 7 as

In [ ]:
count = 19

pbar = tqdm(total=len(PATIENT_PROFILES), desc="Generating synthetic Laboratory Analysis Reports")

for patient in PATIENT_PROFILES:
    print("Processing patient:", patient)
    with open(patient, "r", encoding="utf-8") as f, \
         open(ANALYSIS_PROMPT, "r", encoding="utf-8") as analysis_prompt_file, \
         open(ANALYSIS_SYS_PROMPT, "r", encoding="utf-8") as analysis_sys_prompt_file, \
         open(LAB_EXAMPLE, "r", encoding="utf-8") as lab_example_file, \
         open(f"{OUTPUT_EXP_DIR}_{count}/{PERFECT_OUTPUT_FILE}_{patient.split('_')[2].split('.')[0]}.txt", "r", encoding="utf-8") as perfect_diary_file, \
         open(LAB_SCHEMA, "r", encoding="utf-8") as lab_schema_file:
             
        patient_id = int(patient.split('_')[2].split('.')[0])
        if patient_id in [19, 20]:
            print(f"Generating analysis report for patient {patient_id}")

            patient_data = json.load(f)
        
            base_analysis_prompt = analysis_prompt_file.read()
            analysis_sys_prompt = analysis_sys_prompt_file.read()
            lab_example = lab_example_file.read()
            lab_schema = lab_schema_file.read()
            
            analysis_prompt_w_patient = base_analysis_prompt.replace("{{PATIENT_PROF}}", json.dumps(patient_data))
            analysis_prompt_w_clinical_diary = analysis_prompt_w_patient.replace("{{CLINICAL_DIARY}}", perfect_diary_file.read())
            analysis_prompt_w_lab_schema = analysis_prompt_w_clinical_diary.replace("{{LAB_SCHEMA}}", lab_schema)
            analysis_prompt_final = analysis_prompt_w_lab_schema.replace("{{ANALYSIS_REP}}", lab_example)
            
            
            completion = CLIENT.chat.completions.create(
                model=MODEL, # llama-3.3-70b-versatile, openai/gpt-oss-120b the prompt isn-t optimized for gpt-oss-120b
                messages=[
                    {
                        "role": "system",
                        "content": analysis_sys_prompt
                    },
                    {
                        "role": "user",
                        "content": analysis_prompt_final
                    }
                ],
                temperature=TEMP
            )
            analysis_result = completion.choices[0].message.content
            
            with open(f"{OUTPUT_EXP_DIR}_{count}/{ANALYSIS_OUTPUT_FILE}_{patient_id}.txt","w",encoding="utf-8") as o:
                o.write(f"{analysis_result}\n\n")
                print(f"Saved LLM output on {OUTPUT_EXP_DIR}_{count}/{ANALYSIS_OUTPUT_FILE}_{patient_id}.txt")
                
            output_path = f"{OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt"

            with open(output_path, "a", encoding="utf-8") as o:
                o.write("\n")
                o.write(f"Analysis Report System Prompt:\n{analysis_result}\n")
                o.write("\n")
                o.write(f"Analysis Report Generation Prompt:\n{analysis_prompt_final}\n")
                o.write("\n\n")

                print(f"Saved Analysis report to {OUTPUT_EXP_DIR}_{count}/{TRACK_FILE}_{patient_id}.txt")
            
        else:
            print(f"Skipping patient {patient_id} as it has already been processed.")
        
        pbar.update(1)

Generating synthetic Laboratory Analysis Reports:  83%|████████▎ | 25/30 [00:36<00:07,  1.45s/it]


Processing patient: ./patient_profiles\patient_1.json
Skipping patient 1 as it has already been processed.
Processing patient: ./patient_profiles\patient_10.json
Skipping patient 10 as it has already been processed.
Processing patient: ./patient_profiles\patient_11.json
Generating analysis report for patient 11


Generating synthetic Laboratory Analysis Reports:  10%|█         | 3/30 [00:02<00:25,  1.07it/s]

Saved LLM output on ./outputs/diary-gen_experiment_19/analysis_patient_11.txt
Saved Analysis report to ./outputs/diary-gen_experiment_19/parameter_patient_11.txt
Processing patient: ./patient_profiles\patient_12.json
Skipping patient 12 as it has already been processed.
Processing patient: ./patient_profiles\patient_13.json
Skipping patient 13 as it has already been processed.
Processing patient: ./patient_profiles\patient_14.json
Skipping patient 14 as it has already been processed.
Processing patient: ./patient_profiles\patient_15.json
Skipping patient 15 as it has already been processed.
Processing patient: ./patient_profiles\patient_16.json
Generating analysis report for patient 16


Generating synthetic Laboratory Analysis Reports:  27%|██▋       | 8/30 [00:05<00:15,  1.39it/s]

Saved LLM output on ./outputs/diary-gen_experiment_19/analysis_patient_16.txt
Saved Analysis report to ./outputs/diary-gen_experiment_19/parameter_patient_16.txt
Processing patient: ./patient_profiles\patient_17.json
Generating analysis report for patient 17


Generating synthetic Laboratory Analysis Reports:  30%|███       | 9/30 [00:31<01:41,  4.84s/it]

Saved LLM output on ./outputs/diary-gen_experiment_19/analysis_patient_17.txt
Saved Analysis report to ./outputs/diary-gen_experiment_19/parameter_patient_17.txt
Processing patient: ./patient_profiles\patient_18.json
Skipping patient 18 as it has already been processed.
Processing patient: ./patient_profiles\patient_19.json
Generating analysis report for patient 19


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn0n3ddf6c8a9npft736hda` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 94313, Requested 5841. Please try again in 2m13.055999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}